In [ ]:
# --- Cell 0: Environment check ---
import subprocess, sys, torch

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1024**3:.1f} GB VRAM)")
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: No GPU detected. Generation and training will be slow.")
print(f"Device: {device}")

import random, numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Seeds set.")

In [ ]:
# --- Cell 1: Imports and config ---
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from sr.config import SRConfig

cfg = SRConfig()

print("=== GeneratorConfig ===")
print(f"  image: {cfg.generator.image_width}x{cfg.generator.image_height}")
print(f"  candles: {cfg.generator.num_candles_min}-{cfg.generator.num_candles_max}")
print(f"  zone_magnet_strength: {cfg.generator.zone_magnet_strength}")
print(f"  dark_theme_probability: {cfg.generator.dark_theme_probability}")
print()
print("=== ScenarioWeights ===")
for name, w in zip(cfg.scenarios.names(), cfg.scenarios.weights()):
    print(f"  {name}: {w:.2f}")
print(f"  sum: {sum(cfg.scenarios.weights()):.4f}")
print()
print("=== DatasetConfig ===")
print(f"  num_examples: {cfg.dataset.num_examples}")
print(f"  split: {cfg.dataset.train_frac}/{cfg.dataset.val_frac}/{cfg.dataset.test_frac}")
print()
print("=== ModelConfig ===")
print(f"  base_channels: {cfg.model.base_channels}")
print(f"  out_channels: {cfg.model.out_channels}")
print()
print("=== TrainingConfig ===")
print(f"  epochs: {cfg.training.num_epochs}, batch: {cfg.training.batch_size}")
print(f"  lr: {cfg.training.lr_initial:.2e}, wd: {cfg.training.weight_decay:.2e}")
print(f"  pos_weights: {cfg.training.channel_pos_weights()}")

In [ ]:
# --- Cell 2: Generate dataset ---
from pathlib import Path
from sr.data.generator import generate_dataset

DATA_DIR = Path("data_v3")

generate_dataset(cfg, DATA_DIR, device)

# Verify outputs
import json
jsonl_path = DATA_DIR / "labels_v3.jsonl"
with open(jsonl_path) as f:
    records = [json.loads(line) for line in f]

print(f"\nGenerated {len(records)} examples")

from collections import Counter
sc_counts = Counter(r["scenario"] for r in records)
print("\nScenario distribution:")
for s, c in sorted(sc_counts.items()):
    print(f"  {s}: {c} ({c/len(records)*100:.1f}%)")

import os
n_images = len(list((DATA_DIR / "images").glob("*.png")))
print(f"\nImages saved: {n_images}")

In [ ]:
# --- Cell 3: Preview batch ---
import json, random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from pathlib import Path
from sr.config import ZoneRole
from sr.data.renderer import render_chart_with_zones
from sr.data.labels import compute_heatmap_targets_batch, price_to_pixel_y

DATA_DIR = Path("data_v3")

with open(DATA_DIR / "labels_v3.jsonl") as f:
    records = [json.loads(line) for line in f]

# One per scenario
from collections import defaultdict
from sr.config import ZoneLabel, ZoneRole

def zone_from_dict(d):
    return ZoneLabel(
        zone_id=d["zone_id"], role=ZoneRole(d["role"]),
        low_price=d["low_price"], high_price=d["high_price"],
        center_price=d["center_price"], touch_count=d["touch_count"],
        strength=d["strength"], is_active=d["is_active"],
        center_y=d["center_y"], height_px=d["height_px"],
        confluence_bonus=d.get("confluence_bonus", 0.0),
    )

scenario_to_record = {}
for r in records:
    if r["scenario"] not in scenario_to_record:
        scenario_to_record[r["scenario"]] = r

preview_records = list(scenario_to_record.values())[:11]

fig, axes = plt.subplots(len(preview_records), 2, figsize=(16, 4 * len(preview_records)))

for row, rec in enumerate(preview_records):
    # Left: plain chart
    img_path = DATA_DIR / "images" / f"{rec['id']}.png"
    img = mpimg.imread(str(img_path))
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"{rec['scenario']} | {rec['id']}")
    axes[row, 0].axis("off")
    
    # Right: chart with zones overlay
    import pandas as pd
    ohlc_df = pd.read_csv(DATA_DIR / "ohlc" / f"{rec['id']}.csv")
    ohlc = ohlc_df.values.astype("float32")
    zones = [zone_from_dict(z) for z in rec["zones"]]
    
    tmp_path = f"/tmp/preview_{rec['id']}.png"
    render_chart_with_zones(ohlc, zones, cfg.generator, tmp_path)
    img2 = mpimg.imread(tmp_path)
    axes[row, 1].imshow(img2)
    axes[row, 1].set_title(f"Zones: {len(zones)}")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()
print("Preview complete. If wick clusters are visible at zone boundaries, generation is working correctly.")

In [ ]:
# --- Cell 4: Build dataset and dataloaders ---
from pathlib import Path
from sr.data.dataset import build_dataloaders

DATA_DIR = Path("data_v3")

train_loader, val_loader, test_loader = build_dataloaders(DATA_DIR, cfg, device)

# Verify a batch
batch = next(iter(train_loader))
print(f"Image batch shape: {batch['image'].shape}")
print(f"Target batch shape: {batch['target'].shape}")
print(f"Scenarios: {batch['scenario']}")
print(f"Num zones: {batch['num_zones']}")

# Scenario distribution per split
from collections import Counter
train_scenarios = Counter()
for batch in train_loader:
    train_scenarios.update(batch["scenario"])
print(f"\nTrain scenario distribution (sample):")
for s, c in sorted(train_scenarios.items()):
    print(f"  {s}: {c}")

In [ ]:
# --- Cell 5: Instantiate model and verify ---
from sr.model.net import SupportResistanceHeatmapNetV3
from sr.model.loss import SoftHeatmapLossV3
import matplotlib.pyplot as plt

model = SupportResistanceHeatmapNetV3(dropout=cfg.model.dropout).to(device)
print(f"Model parameters: {model.count_parameters():,}")

# One forward pass
batch = next(iter(val_loader))
images = batch["image"].to(device)
targets = batch["target"].to(device)

with torch.no_grad():
    logits = model(images)
    preds = torch.sigmoid(logits)

print(f"Input shape:  {images.shape}")
print(f"Output shape: {logits.shape}")
assert logits.shape == (images.shape[0], 5, cfg.model.output_height), f"Unexpected output shape: {logits.shape}"
print("Shape assertion passed.")

# Loss test
loss_fn = SoftHeatmapLossV3(cfg.training).to(device)
loss_dict = loss_fn(logits, targets)
print(f"Loss: {loss_dict['loss'].item():.4f} | BCE: {loss_dict['bce'].item():.4f} | PeakMSE: {loss_dict['peak_mse'].item():.4f}")

# Plot untrained predictions vs targets for first example
fig, axes = plt.subplots(2, 5, figsize=(18, 5))
channel_names = ["Support", "Resistance", "Active", "Historical", "Proximity"]
for c, name in enumerate(channel_names):
    axes[0, c].plot(targets[0, c].cpu().numpy())
    axes[0, c].set_title(f"GT: {name}")
    axes[0, c].set_ylim(0, 1)
    axes[1, c].plot(preds[0, c].cpu().numpy())
    axes[1, c].set_title(f"Pred: {name} (untrained)")
    axes[1, c].set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 6: Train ---
from pathlib import Path
from sr.train import loop

DATA_DIR = Path("data_v3")

# Re-instantiate model fresh for training
model = SupportResistanceHeatmapNetV3(dropout=cfg.model.dropout).to(device)
print(f"Model parameters: {model.count_parameters():,}")

history = loop.run(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
    output_dir=DATA_DIR,
)

# Plot training curves
import pandas as pd
import matplotlib.pyplot as plt

hist_df = pd.read_csv(DATA_DIR / "training_history_v3.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="train")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(hist_df["epoch"], hist_df["train_recall"], label="train")
axes[1].plot(hist_df["epoch"], hist_df["val_recall"], label="val")
axes[1].set_title("Zone Recall @ 10px")
axes[1].legend()

axes[2].plot(hist_df["epoch"], hist_df["lr"])
axes[2].set_title("Learning Rate")
axes[2].set_yscale("log")

plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 7: Evaluate ---
import torch
from pathlib import Path
from sr.model.net import SupportResistanceHeatmapNetV3
from sr.train import metrics as sr_metrics
from sr.train.loop import eval_one_epoch
from sr.model.loss import SoftHeatmapLossV3
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

DATA_DIR = Path("data_v3")
ckpt_path = DATA_DIR / "checkpoints" / "best.pt"

# Load best checkpoint
ckpt = torch.load(ckpt_path, map_location=device)
model = SupportResistanceHeatmapNetV3(dropout=cfg.model.dropout).to(device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")

loss_fn = SoftHeatmapLossV3(cfg.training).to(device)

test_metrics = eval_one_epoch(
    model, test_loader, loss_fn, device,
    epoch=ckpt["epoch"],
    save_worst_dir=DATA_DIR / "worst_recall",
)

print("\n=== Test Set Metrics ===")
print(f"  Loss:              {test_metrics['loss']:.4f}")
print(f"  Zone Recall @10px: {test_metrics['zone_recall_10px']:.4f}")
print(f"  False Peak Rate:   {test_metrics['false_peak_rate']:.4f}")

channel_names = ["Support", "Resistance", "Active", "Historical", "Proximity"]
print("\n  Per-channel Recall @10px:")
for name, r in zip(channel_names, test_metrics["per_channel_recall_10px"]):
    print(f"    {name}: {r:.4f}")

print("\n  Per-channel MAE (px):")
for name, m in zip(channel_names, test_metrics["per_channel_mae_px"]):
    val = f"{m:.1f}" if not (m != m) else "N/A"
    print(f"    {name}: {val}")

# Show worst-recall examples
worst_file = DATA_DIR / "worst_recall" / f"worst_recall_epoch_{ckpt['epoch']}.txt"
if worst_file.exists():
    worst_ids = worst_file.read_text().splitlines()[:6]
    if worst_ids:
        fig, axes = plt.subplots(1, len(worst_ids), figsize=(4 * len(worst_ids), 4))
        if len(worst_ids) == 1:
            axes = [axes]
        for ax, img_id in zip(axes, worst_ids):
            img_path = DATA_DIR / "images" / f"{img_id}.png"
            if img_path.exists():
                ax.imshow(mpimg.imread(str(img_path)))
            ax.set_title(img_id[:8])
            ax.axis("off")
        plt.suptitle("Worst Recall Examples")
        plt.tight_layout()
        plt.show()

In [ ]:
# --- Cell 8: Real ticker inference ---
import yfinance as yf
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
import tempfile
from pathlib import Path
from sr.model.net import SupportResistanceHeatmapNetV3
from sr.data.renderer import render_chart
from sr.data.labels import pixel_y_to_price
from sr.config import SRConfig

DATA_DIR = Path("data_v3")
cfg = SRConfig()

# Load best checkpoint
ckpt = torch.load(DATA_DIR / "checkpoints" / "best.pt", map_location=device)
model = SupportResistanceHeatmapNetV3(dropout=cfg.model.dropout).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

def get_ticker_levels_v3(ticker: str, period: str = "6mo", sensitivity: str = "balanced"):
    """Run S/R inference on a real ticker."""
    df = yf.download(ticker, period=period, interval="1d", progress=False)
    if df.empty:
        print(f"No data for {ticker}")
        return
    
    ohlc = df[["Open", "High", "Low", "Close"]].values.astype("float32")
    
    # Render chart to temp file
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
        tmp_path = f.name
    
    render_chart(ohlc, cfg.generator, tmp_path, dark_theme=False, draw_grid=True, draw_axes=False)
    
    # Load and preprocess
    from PIL import Image
    import torchvision.transforms.functional as TF
    img = Image.open(tmp_path).convert("RGB")
    img_tensor = TF.to_tensor(img).unsqueeze(0).to(device)  # [1, 3, H, W]
    
    with torch.no_grad():
        logits = model(img_tensor)
        preds = torch.sigmoid(logits)[0]  # [5, H]
    
    # Extract levels
    from sr.train.metrics import find_peaks_1d
    
    price_min = ohlc[:, 2].min() * 0.99
    price_max = ohlc[:, 1].max() * 1.01
    
    profile = cfg.inference.sensitivity_profiles[sensitivity]
    min_score = profile["min_score"]
    
    channel_names = ["Support", "Resistance", "Active", "Historical", "Proximity"]
    levels = []
    for ch in range(3):  # support, resistance, active only
        peaks = find_peaks_1d(preds[ch].cpu(), threshold=min_score)
        for pk in peaks:
            price = pixel_y_to_price(pk, price_min, price_max, cfg.generator.image_height)
            score = preds[ch, pk].item()
            levels.append({
                "channel": channel_names[ch],
                "price": price,
                "score": score,
                "pixel_y": pk,
            })
    
    levels.sort(key=lambda x: x["price"])
    
    print(f"\n{ticker} ({sensitivity}) — {len(levels)} levels found:")
    print(f"{'Channel':<12} {'Price':>8} {'Score':>6}")
    print("-" * 30)
    for lv in levels:
        print(f"{lv['channel']:<12} ${lv['price']:>7.2f} {lv['score']:>6.3f}")
    
    # Display annotated chart
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    img_arr = mpimg.imread(tmp_path)
    axes[0].imshow(img_arr)
    for lv in levels:
        color = "green" if lv["channel"] == "Support" else "red" if lv["channel"] == "Resistance" else "yellow"
        axes[0].axhline(y=lv["pixel_y"], color=color, alpha=0.7, linewidth=1.5)
    axes[0].set_title(f"{ticker} — annotated levels")
    axes[0].axis("off")
    
    # Plot heatmap channels
    axes[1].plot(preds[0].cpu().numpy(), label="Support", color="green", alpha=0.8)
    axes[1].plot(preds[1].cpu().numpy(), label="Resistance", color="red", alpha=0.8)
    axes[1].plot(preds[2].cpu().numpy(), label="Active", color="yellow", alpha=0.8)
    axes[1].set_title("Predicted Heatmaps")
    axes[1].legend()
    axes[1].set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
    
    return levels

# Run on test tickers
test_tickers = ["AAPL", "SPY", "TSLA"]
for ticker in test_tickers:
    get_ticker_levels_v3(ticker, sensitivity="balanced")